In [0]:
CREATE OR REPLACE MATERIALIZED VIEW workspace.default.dividend_best_match_mv
TRIGGER ON UPDATE AT MOST EVERY INTERVAL 1 MINUTES
AS

WITH dividend_base AS (
  SELECT
    d.*,
    lower(trim(regexp_extract(d.transaction_details, '/D/([^|]+)', 1))) AS extracted_party,
    lower(d.transaction_details) AS transaction_details_lc
  FROM workspace.default.dividend d
),

dividend_tokens AS (
  SELECT
    db.transaction_details,
    token AS d_token
  FROM dividend_base db
  LATERAL VIEW explode(
    filter(
      split(regexp_replace(db.extracted_party, '[^a-z0-9]+', ' '), ' '),
      x -> length(x) >= 3
           AND x NOT IN ('berhad','bhd','sdn','holding','holdings','industry','industries','dividend','payment','ibg','credit','interbank','giro','autopay','transfer')
    )
  ) t AS token
),
portfolio_all AS (
  SELECT name,symbol FROM workspace.default.cgs_portfolio_yahoo
  UNION ALL
  SELECT name,stock_name AS symbol FROM workspace.default.hlebroking_portfolio_yahoo
),
portfolio_symbol_base AS (
  SELECT
    p.name,
    p.symbol,
    trim(regexp_replace(lower(p.symbol), '\\.kl$', '')) AS symbol_clean
  FROM portfolio_all p
  WHERE p.symbol IS NOT NULL
),
portfolio_all_base AS (
  SELECT name,symbol FROM workspace.default.cgs_portfolio_yahoo
  UNION ALL
  SELECT name,stock_name AS symbol FROM workspace.default.hlebroking_portfolio_yahoo
),
portfolio_alias_tokens AS (
  SELECT
    p.name,
    p.symbol,
    'name' AS match_source,
    token AS p_token
  FROM portfolio_all_base p
  LATERAL VIEW explode(
    filter(
      split(regexp_replace(lower(p.name), '[^a-z0-9]+', ' '), ' '),
      x -> length(x) >= 3
           AND x NOT IN ('berhad','bhd','sdn','holding','holdings','industry','industries' ,'ibg','credit','interbank','giro','autopay','transfer')
    )
  ) t AS token

  UNION ALL

  SELECT
    ps.name,
    ps.symbol,
    'symbol' AS match_source,
    ps.symbol_clean AS p_token
  FROM portfolio_symbol_base ps
  WHERE length(ps.symbol_clean) >= 2
),

candidate_matches_token AS (
  SELECT
    dt.transaction_details,
    pt.name,
    pt.symbol,
    SUM(
      CASE
        WHEN pt.match_source = 'symbol' AND dt.d_token = pt.p_token THEN 6
        WHEN pt.match_source = 'symbol'
             AND length(pt.p_token) >= 4
             AND levenshtein(dt.d_token, pt.p_token, 1) >= 0 THEN 4
        WHEN pt.match_source = 'name' AND dt.d_token = pt.p_token THEN 3
        WHEN pt.match_source = 'name'
             AND length(dt.d_token) >= 5
             AND length(pt.p_token) >= 5
             AND levenshtein(dt.d_token, pt.p_token, 1) >= 0 THEN 2
        ELSE 0
      END
    ) AS match_score,
    COUNT(*) AS matched_token_rows,
    MAX(CASE WHEN pt.match_source = 'symbol' THEN 1 ELSE 0 END) AS matched_by_symbol,
    0 AS matched_by_symbol_contains
  FROM dividend_tokens dt
  JOIN portfolio_alias_tokens pt
    ON (pt.match_source = 'symbol' AND (
          dt.d_token = pt.p_token
          OR (length(pt.p_token) >= 4 AND levenshtein(dt.d_token, pt.p_token, 1) >= 0)
       ))
    OR (pt.match_source = 'name' AND (
          dt.d_token = pt.p_token
          OR (
               length(dt.d_token) >= 5
               AND length(pt.p_token) >= 5
               AND levenshtein(dt.d_token, pt.p_token, 1) >= 0
             )
       ))
  GROUP BY
    dt.transaction_details,
    pt.name,
    pt.symbol
),

candidate_matches_symbol_contains AS (
  SELECT
    db.transaction_details,
    ps.name,
    ps.symbol,
    CASE
      WHEN instr(db.transaction_details_lc, ps.symbol_clean) > 0 THEN 8
      ELSE 0
    END AS match_score,
    0 AS matched_token_rows,
    1 AS matched_by_symbol,
    1 AS matched_by_symbol_contains
  FROM dividend_base db
  JOIN portfolio_symbol_base ps
    ON length(ps.symbol_clean) >= 3
   AND instr(db.transaction_details_lc, ps.symbol_clean) > 0
),

candidate_matches_all AS (
  SELECT * FROM candidate_matches_token
  UNION ALL
  SELECT * FROM candidate_matches_symbol_contains
),

candidate_matches_agg AS (
  SELECT
    transaction_details,
    name,
    symbol,
    SUM(match_score) AS match_score,
    SUM(matched_token_rows) AS matched_token_rows,
    MAX(matched_by_symbol) AS matched_by_symbol,
    MAX(matched_by_symbol_contains) AS matched_by_symbol_contains
  FROM candidate_matches_all
  GROUP BY
    transaction_details,
    name,
    symbol
),

best_match AS (
  SELECT *
  FROM candidate_matches_agg
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY transaction_details
    ORDER BY
      match_score DESC,
      matched_by_symbol_contains DESC,
      matched_by_symbol DESC,
      matched_token_rows DESC,
      length(name) DESC
  ) = 1
)

SELECT
  d.*,
  b.name   AS matched_name,
  b.symbol AS matched_symbol,
  b.match_score,
  b.matched_by_symbol,
  b.matched_by_symbol_contains,
  b.matched_token_rows
FROM workspace.default.dividend d
LEFT JOIN best_match b
  ON d.transaction_details = b.transaction_details;
